# Reproducible label-noise experiments

This notebook is the single entry point for constructing the canonical actual-prediction and perfect-predictor databases. Its scientific parameters live in `config/experiments.toml`; execution is deterministic and resumable.

In [1]:
import dataclasses
import pathlib
import sys

import pandas as pd

PROJECT_ROOT = pathlib.Path.cwd().resolve()
if PROJECT_ROOT.name == 'src':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from experiment_pipeline import (
    build_experiment_plan,
    database_summary,
    evaluate_experiments,
    generate_experiments,
    load_experiment_config,
    validate_experiment_inputs,
)

## 1. Load the experiment configuration

In [2]:
CONFIG_PATH = PROJECT_ROOT / 'config' / 'experiments.toml'
config = load_experiment_config(CONFIG_PATH)
config.raw

{'project': {'root': '..'},
 'dataset': {'data_root': 'data/oem_sar',
  'labels_dir': 'test/labels',
  'labels_pattern': '*.tif',
  'predictions_dir': 'results',
  'predictions_pattern': '*.png',
  'expected_images': 490,
  'num_classes': 9},
 'outputs': {'noise_dir': 'data/oem_sar/noise',
  'actual_db': 'experiments/eval_results.db',
  'perfect_db': 'experiments/eval_results_perfect_pred.db',
  'manifest': 'experiments/experiment_manifest.json',
  'backup_dir': 'experiments/backups'},
 'run': {'global_seed': 42,
  'repetitions': 5,
  'resume': True,
  'adopt_legacy': False},
 'inference': {'enabled': False,
  'checkpoint': 'external/oem_sar/Semantic_Segmentation/pretrained/SAR_Mix_5_u-efficientnet-b4.pth',
  'results_dir': 'results'},
 'noise': [{'method': 'random_pixels',
   'severity_parameter': 'noise_rate',
   'values': [0.025, 0.05, 0.075, 0.1, 0.125, 0.15, 0.175, 0.2, 0.225, 0.25],
   'params': {'n_classes': 9}},
  {'method': 'swap_classes',
   'severity_parameter': 'swap_prob',

## 2. Validate immutable inputs

Inference is disabled by default. If enabled in the TOML file, the pipeline validates the checkpoint and creates predictions before the final validation.

In [3]:
validation = validate_experiment_inputs(config)
pd.Series(dataclasses.asdict(validation), name='value')

/home/schroom/Projects/ba_thesis/.venv/lib/python3.12/site-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


reference_files                490
prediction_files               490
image_shape           (1024, 1024)
inference_required           False
Name: value, dtype: object

## 3. Preview the canonical plan

In [4]:
plan = build_experiment_plan(config)
plan_frame = pd.DataFrame(
    {
        'method': item.method,
        'params': item.params,
        'repetition': item.repetition,
        'seed': item.seed,
        'fingerprint': item.fingerprint,
    }
    for item in plan
)
display(plan_frame.groupby('method').size().rename('planned_runs'))
plan_frame.head()

method
add_omission           50
boundary_morphology    50
random_pixels          50
shift_objects          50
shift_scene            50
swap_classes           50
zoom                   50
Name: planned_runs, dtype: int64

,method,params,repetition,seed,fingerprint
0,random_pixels,"{'n_classes': 9, 'noise_rate': 0.025}",0,140436573,827b7d2a90c6343d5407
1,random_pixels,"{'n_classes': 9, 'noise_rate': 0.025}",1,2870118025,a7c3c9c7168dd1648b05
2,random_pixels,"{'n_classes': 9, 'noise_rate': 0.025}",2,2112289069,caa2478adcbe44052e2f
3,random_pixels,"{'n_classes': 9, 'noise_rate': 0.025}",3,3420408022,88bad072591a2f07ce69
4,random_pixels,"{'n_classes': 9, 'noise_rate': 0.025}",4,3359749232,a449d1abc044b05dc7c5


## 4. Generate or adopt noisy datasets

This phase creates and verifies all noisy masks. It records generation metadata in working databases but does not calculate metrics or replace the active databases.

In [5]:
generation_summary = generate_experiments(config)
pd.Series(dataclasses.asdict(generation_summary), name='value')

/home/schroom/Projects/ba_thesis/.venv/lib/python3.12/site-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


planned                                                          350
adopted                                                            0
generated                                                          0
resumed                                                          350
manifest           /home/schroom/Projects/ba_thesis/experiments/e...
actual_work_db     /home/schroom/Projects/ba_thesis/experiments/e...
perfect_work_db    /home/schroom/Projects/ba_thesis/experiments/e...
backups            (/home/schroom/Projects/ba_thesis/experiments/...
Name: value, dtype: object

## 5. Evaluate generated datasets

This separate phase calculates actual-prediction and perfect-predictor metrics. The active databases are replaced only after the complete pair passes validation.

In [5]:
evaluation_summary = evaluate_experiments(config)
pd.Series(dataclasses.asdict(evaluation_summary), name='value')

/home/schroom/Projects/ba_thesis/.venv/lib/python3.12/site-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


planned                                                            350
actual_evaluated                                                   231
perfect_evaluated                                                  231
actual_db            /home/schroom/Projects/ba_thesis/experiments/e...
perfect_db           /home/schroom/Projects/ba_thesis/experiments/e...
manifest             /home/schroom/Projects/ba_thesis/experiments/e...
Name: value, dtype: object

## 6. Verify final database completeness

In [ ]:
summary_frame = pd.DataFrame(database_summary(config))
display(summary_frame)
assert len(plan) == 350
assert (summary_frame['rows'] == 50).all()
assert (summary_frame['complete_rows'] == summary_frame['rows']).all()
print('Both canonical databases are complete.')